# LeetCode #147 — Insertion Sort List: sắp xếp chèn trên danh sách liên kết

**Mức độ:** Medium · **Chủ đề:** Linked List, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def insertionSortList(self, head: Optional[ListNode]) -> Optional[ListNode]:
```

Mục tiêu của notebook:

1. Ôn **sắp xếp chèn (insertion sort)** trên mảng.
2. Chuyển sang danh sách liên kết: không đi lùi được → tìm vị trí chèn **từ đầu** danh sách đã sắp xếp.
3. Dùng **nút giả** làm đầu danh sách kết quả; tối ưu "nếu nút mới ≥ đuôi thì gắn luôn".
4. Phân tích: `O(n²)` xấu nhất, `O(n)` khi gần như đã sắp xếp; ổn định (stable).

# 1. Bài toán là gì?

Cho `head` của danh sách liên kết, sắp xếp tăng dần bằng **sắp xếp chèn**, trả về đầu danh sách mới.

Sắp xếp chèn: lần lượt lấy từng phần tử từ đầu vào, chèn vào đúng vị trí trong phần **đã sắp xếp**.

**Constraints (theo đề):**

- Số nút trong khoảng `[1, 5000]`.
- `-5000 <= Node.val <= 5000`

---

## Ví dụ

```text
4 → 2 → 1 → 3          ->  1 → 2 → 3 → 4
-1 → 5 → 3 → 4 → 0     ->  -1 → 0 → 3 → 4 → 5
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Sắp xếp (sorting)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [6]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [7]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [8]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.3 Áp dụng vào bài này

**Sắp xếp chèn** (như trên mảng) nhưng trên **danh sách liên kết**: không đi lùi được nên tìm vị trí chèn bằng cách đi **xuôi** từ đầu phần đã sắp xếp (bắt đầu bằng **nút giả**), và chèn bằng đổi con trỏ.

# 3. Nền tảng: insertion sort trên mảng

In [9]:
from typing import List, Optional
import random, time


def insertion_sort_array(a: List[int]) -> List[int]:
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:        # dịch các phần tử lớn hơn sang phải
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = x
    return a

print(insertion_sort_array([4, 2, 1, 3]), insertion_sort_array([-1, 5, 3, 4, 0]))

[1, 2, 3, 4] [-1, 0, 3, 4, 5]


Trên mảng, ta tìm vị trí chèn bằng cách đi **lùi** từ `i − 1`. Danh sách liên kết đơn **không** đi lùi được → phải tìm vị trí chèn bằng cách đi **xuôi** từ đầu phần đã sắp xếp.

In [10]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head, limit=10**6):
    out = []
    while head and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out

# 4. Ý tưởng chính

Giữ danh sách **đã sắp xếp** bắt đầu từ nút giả `dummy`. Lần lượt tách từng nút `cur` khỏi danh sách gốc và chèn vào:

```text
dummy = ListNode()
cur = head
while cur:
    nxt = cur.next                  # lưu trước khi tách
    prev = dummy
    while prev.next and prev.next.val <= cur.val:   # tìm chỗ chèn
        prev = prev.next
    cur.next = prev.next            # chèn sau prev
    prev.next = cur
    cur = nxt
return dummy.next
```

`<=` (không phải `<`) giữ **ổn định**: nút có giá trị bằng được chèn **sau** các nút bằng đã có.

In [11]:
def insertion_sort_basic(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode()
    cur = head
    while cur:
        nxt = cur.next
        prev = dummy
        while prev.next and prev.next.val <= cur.val:
            prev = prev.next
        cur.next = prev.next
        prev.next = cur
        cur = nxt
    return dummy.next

print(to_list(insertion_sort_basic(build_list([4, 2, 1, 3]))), to_list(insertion_sort_basic(build_list([-1, 5, 3, 4, 0]))))

[1, 2, 3, 4] [-1, 0, 3, 4, 5]


# 5. Mô phỏng

In [12]:
def trace_insertion(values):
    head = build_list(values)
    dummy = ListNode("D")
    cur = head
    while cur:
        nxt = cur.next
        prev, steps = dummy, 0
        while prev.next and prev.next.val <= cur.val:
            prev = prev.next
            steps += 1
        cur.next = prev.next
        prev.next = cur
        where = "đầu" if prev is dummy else f"sau {prev.val}"
        print(f"chèn {cur.val:>2} vào {where:<7} ({steps} bước tìm) -> đã sắp xếp: {to_list(dummy.next)} | còn lại: {to_list(nxt)}")
        cur = nxt
    return to_list(dummy.next)

_ = trace_insertion([-1, 5, 3, 4, 0])

chèn -1 vào đầu     (0 bước tìm) -> đã sắp xếp: [-1] | còn lại: [5, 3, 4, 0]


chèn  5 vào sau -1  (1 bước tìm) -> đã sắp xếp: [-1, 5] | còn lại: [3, 4, 0]
chèn  3 vào sau -1  (1 bước tìm) -> đã sắp xếp: [-1, 3, 5] | còn lại: [4, 0]
chèn  4 vào sau 3   (2 bước tìm) -> đã sắp xếp: [-1, 3, 4, 5] | còn lại: [0]
chèn  0 vào sau -1  (1 bước tìm) -> đã sắp xếp: [-1, 0, 3, 4, 5] | còn lại: []


# 6. Tối ưu: gắn vào đuôi khi không cần tìm

Nếu dữ liệu **gần như đã sắp xếp**, hầu hết các nút đều ≥ phần tử lớn nhất hiện có — bản cơ bản vẫn phải đi từ đầu tới cuối mỗi lần (`O(n²)` cho cả mảng đã sắp xếp!).

Cải tiến: giữ con trỏ `tail` (nút cuối của phần đã sắp xếp). Nếu `cur.val >= tail.val` → gắn ngay sau `tail`, `O(1)`.

Cải tiến thứ hai (cũng hay dùng): giữ `prev` của lần chèn trước; nếu `cur.val` ≥ `prev.val` thì bắt đầu tìm từ `prev` thay vì từ đầu.

In [13]:
def insertion_sort_tail(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(float("-inf"))
    tail = dummy
    cur = head
    while cur:
        nxt = cur.next
        if cur.val >= tail.val:                    # gắn vào đuôi
            tail.next = cur
            tail = cur
            tail.next = None
        else:
            prev = dummy
            while prev.next.val <= cur.val:        # chắc chắn dừng trước tail vì cur.val < tail.val
                prev = prev.next
            cur.next = prev.next
            prev.next = cur
        cur = nxt
    return dummy.next

print(to_list(insertion_sort_tail(build_list([4, 2, 1, 3]))), to_list(insertion_sort_tail(build_list([-1, 5, 3, 4, 0]))))

[1, 2, 3, 4] [-1, 0, 3, 4, 5]


In [14]:
def count_compare(fn, values):
    """Đếm số lần so sánh trong vòng tìm vị trí (đo bằng cách bọc val)."""
    count = 0
    class V(int):
        def __le__(self, other):
            nonlocal count
            count += 1
            return int(self) <= other
        def __ge__(self, other):
            nonlocal count
            count += 1
            return int(self) >= other
    fn(build_list([V(v) for v in values]))
    return count

sorted_input = list(range(2000))
print("Đã sắp xếp 2000 phần tử — số lần so sánh:",
      "bản cơ bản =", count_compare(insertion_sort_basic, sorted_input),
      "| bản có tail =", count_compare(insertion_sort_tail, sorted_input))

Đã sắp xếp 2000 phần tử — số lần so sánh: bản cơ bản = 3998000 | bản có tail = 3999


# 7. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def insertionSortList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode(float("-inf"))
        tail = dummy
        cur = head
        while cur:
            nxt = cur.next
            if cur.val >= tail.val:
                tail.next = cur
                tail = cur
                tail.next = None
            else:
                prev = dummy
                while prev.next.val <= cur.val:
                    prev = prev.next
                cur.next = prev.next
                prev.next = cur
            cur = nxt
        return dummy.next

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def run(f, values):
    head = build_list(values)
    ids = set()
    n = head
    while n:
        ids.add(id(n)); n = n.next
    out = f(head)
    res, got_ids, n = [], set(), out
    while n and len(res) <= len(values):
        res.append(n.val); got_ids.add(id(n)); n = n.next
    assert len(res) == len(values) and got_ids == ids, "phải dùng lại đúng các nút, không vòng"
    return res


sol = Solution()
check("Ví dụ 1", run(sol.insertionSortList, [4, 2, 1, 3]), [1, 2, 3, 4])
check("Ví dụ 2", run(sol.insertionSortList, [-1, 5, 3, 4, 0]), [-1, 0, 3, 4, 5])
check("Một nút", run(sol.insertionSortList, [7]), [7])
check("Đã sắp xếp", run(sol.insertionSortList, [1, 2, 3]), [1, 2, 3])
check("Giảm dần", run(sol.insertionSortList, [3, 2, 1]), [1, 2, 3])
check("Trùng", run(sol.insertionSortList, [2, 1, 2, 1]), [1, 1, 2, 2])

random.seed(147)
for _ in range(3000):
    vals = [random.randint(-5, 5) for _ in range(random.randint(1, 20))]
    assert run(sol.insertionSortList, vals) == sorted(vals) == run(insertion_sort_basic, vals)
print("3000 danh sách ngẫu nhiên: khớp sorted()")

for name, vals in [("ngẫu nhiên", [random.randint(-5000, 5000) for _ in range(5000)]),
                   ("đã sắp xếp", list(range(5000))),
                   ("giảm dần", list(range(5000, 0, -1))),
                   ("xấu nhất", [0, 10**4] + list(range(1, 4999)))]:
    t0 = time.time(); run(sol.insertionSortList, vals); t1 = time.time()
    print(f"5000 nút {name:<11}: {t1 - t0:.3f}s")

Ví dụ 1          PASS  [1, 2, 3, 4]
Ví dụ 2          PASS  [-1, 0, 3, 4, 5]
Một nút          PASS  [7]
Đã sắp xếp       PASS  [1, 2, 3]
Giảm dần         PASS  [1, 2, 3]
Trùng            PASS  [1, 1, 2, 2]


3000 danh sách ngẫu nhiên: khớp sorted()

5000 nút ngẫu nhiên : 0.893s
5000 nút đã sắp xếp : 0.003s
5000 nút giảm dần   : 0.012s


5000 nút xấu nhất   : 1.392s


Trên danh sách liên kết (tìm từ đầu), dãy **giảm dần** lại là trường hợp **tốt**: mỗi nút mới nhỏ nhất, chèn ngay sau `dummy`. Trường hợp xấu là khi mỗi nút mới phải đi gần hết danh sách nhưng **không** được gắn vào đuôi — như `[0, 10000, 1, 2, …]`: phần tử `10000` làm đuôi, mọi số sau đều nhỏ hơn đuôi nhưng lớn hơn mọi số trước.

Kiểm tra **ổn định**: hai nút cùng giá trị giữ thứ tự ban đầu.

In [17]:
head = build_list([2, 1, 2, 1])
a1, b1 = head.next, head.next.next.next       # hai nút giá trị 1, theo thứ tự ban đầu
a2, b2 = head, head.next.next                 # hai nút giá trị 2
out = sol.insertionSortList(head)
order = []
n = out
while n:
    order.append(n); n = n.next
print("Ổn định?", order.index(a1) < order.index(b1) and order.index(a2) < order.index(b2))

Ổn định? True


# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** trước mỗi vòng, danh sách từ `dummy.next` gồm đúng các nút đã xử lý, theo thứ tự tăng dần, và `tail` là nút cuối của nó.

- Nếu `cur.val ≥ tail.val`: gắn sau `tail` vẫn tăng dần.
- Ngược lại: vòng tìm dừng ở `prev` cuối cùng có `prev.val ≤ cur.val` (luôn dừng trước `tail` vì `tail.val > cur.val`) → chèn giữa `prev` và `prev.next` vẫn tăng dần. `tail` không đổi.

```text
Time:  O(n²) xấu nhất; O(n) nếu đã sắp xếp (nhờ tail) hoặc giảm dần (mỗi nút chèn ngay đầu)
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Không lưu `cur.next` trước khi chèn

In [18]:
def wrong_no_save(head):
    dummy, cur, steps = ListNode(), head, 0
    while cur and steps < 50:
        prev = dummy
        while prev.next and prev.next.val <= cur.val:
            prev = prev.next
        cur.next = prev.next                       # ghi đè cur.next…
        prev.next = cur
        cur = cur.next                             # SAI: …rồi lại đi theo nó
        steps += 1
    return to_list(dummy.next, 10), steps

print("Sai :", wrong_no_save(build_list([4, 2, 1, 3])))

Sai : ([4], 1)


Sau khi chèn, `cur.next` đã bị đổi thành nút **kế tiếp trong danh sách đã sắp xếp** (ở lần đầu là `None`) → vòng lặp dừng ngay, **mất** toàn bộ `2 → 1 → 3`. Với dữ liệu khác, `cur` có thể đi vào chính danh sách kết quả và lặp mãi (vì vậy ta chặn ở 50 bước).

## Lỗi 2: Dùng `<` thay vì `<=` (mất ổn định)

Kết quả vẫn **đúng thứ tự giá trị**, nhưng các nút bằng nhau bị **đảo** thứ tự ban đầu — ảnh hưởng khi nút mang thêm dữ liệu (sắp xếp theo khóa).

In [19]:
def basic_with(cmp_strict, head):
    dummy, cur = ListNode(), head
    while cur:
        nxt, prev = cur.next, dummy
        while prev.next and (prev.next.val < cur.val if cmp_strict else prev.next.val <= cur.val):
            prev = prev.next
        cur.next, prev.next = prev.next, cur
        cur = nxt
    return dummy.next

head = build_list([1, 1]); first = head
out = basic_with(True, head)
print("Dùng '<'  : nút 1 đầu tiên ban đầu giờ đứng thứ", [n for n in [out, out.next]].index(first) + 1)
head = build_list([1, 1]); first = head
out = basic_with(False, head)
print("Dùng '<=' : nút 1 đầu tiên ban đầu giờ đứng thứ", [n for n in [out, out.next]].index(first) + 1)

Dùng '<'  : nút 1 đầu tiên ban đầu giờ đứng thứ 2
Dùng '<=' : nút 1 đầu tiên ban đầu giờ đứng thứ 1


## Lỗi 3: Bản có `tail` nhưng quên `tail.next = None`

Nút vừa gắn vào đuôi vẫn trỏ tới phần chưa xử lý → danh sách kết quả "dính" vào danh sách gốc, có thể tạo vòng.

## Lỗi 4: Sắp xếp bằng cách đổi **giá trị** (bubble sort trên `val`)

Không phải sắp xếp chèn, và không dùng lại cấu trúc nút như đề mong muốn. Với 5000 nút, `O(n²)` đổi giá trị cũng chậm.

# 10. Mẹo nhận diện pattern

- **Chèn vào danh sách liên kết đã sắp xếp** → nút giả + đi xuôi tìm `prev` cuối cùng `≤ x`.
- **Sắp xếp danh sách liên kết hiệu quả** → merge sort `O(n log n)` (#148) — insertion sort chỉ tốt khi dữ liệu gần sắp xếp.

Bài liên quan: **#148 Sort List**, **#21 Merge Two Sorted Lists**, **#708 Insert into a Sorted Circular Linked List**, **#83** / **#82** (danh sách đã sắp xếp).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Mô phỏng insertion sort trên `5 → 1 → 4 → 2 → 3`; với mỗi nút, cần bao nhiêu bước tìm (bản cơ bản)?

## Bài 2 — Điền chỗ trống (bản cơ bản)

In [20]:
def exercise_2_insertion_sort(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode()
    cur = head
    while cur:
        # TODO: lưu nxt; tìm prev; chèn cur sau prev; cur = nxt
        pass
    return dummy.next


# assert to_list(exercise_2_insertion_sort(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
# assert to_list(exercise_2_insertion_sort(build_list([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]

## Bài 3 — Biến thể: chèn một giá trị vào danh sách đã sắp xếp

Viết `insert_sorted(head, x)` trả về đầu danh sách sau khi chèn nút mới giá trị `x` (giữ tăng dần).

In [21]:
def exercise_3_insert_sorted(head: Optional[ListNode], x: int) -> ListNode:
    # TODO
    pass


# assert to_list(exercise_3_insert_sorted(build_list([1, 3, 5]), 4)) == [1, 3, 4, 5]
# assert to_list(exercise_3_insert_sorted(build_list([1, 3, 5]), 0)) == [0, 1, 3, 5]
# assert to_list(exercise_3_insert_sorted(None, 7)) == [7]

## Bài 4 — Biến thể: Insert into a Sorted Circular Linked List (LeetCode #708)

Danh sách **vòng** đã sắp xếp tăng (nhưng `head` có thể là nút bất kỳ). Chèn `x` sao cho vẫn sắp xếp vòng. Trả về `head` (hoặc nút mới nếu danh sách rỗng).

Gợi ý nhanh: đi vòng tìm cặp `(p, p.next)` với `p.val <= x <= p.next.val`, hoặc chỗ "gãy" (`p.val > p.next.val`) khi `x` lớn hơn max hoặc nhỏ hơn min.

In [22]:
def exercise_4_insert_circular(head: Optional[ListNode], x: int) -> ListNode:
    # TODO
    pass


def circ_to_list(start, k):
    out, n = [], start
    for _ in range(k):
        out.append(n.val); n = n.next
    return out

# h = build_list([3, 4, 1]); h.next.next.next = h; r = exercise_4_insert_circular(h, 2); assert sorted(circ_to_list(r, 4)) == [1, 2, 3, 4] and circ_to_list(r, 4) == [3, 4, 1, 2]
# r = exercise_4_insert_circular(None, 1); assert r.next is r and r.val == 1

## Bài 5 — Mini challenge: đếm số nghịch thế trong lúc sắp xếp chèn

Số bước **dịch chuyển** của insertion sort trên mảng bằng đúng số **nghịch thế** (cặp `i < j` với `a[i] > a[j]`). Trên danh sách liên kết, đếm số nút **lớn hơn** `cur` đã có trong phần đã sắp xếp mỗi khi chèn — tổng chính là số nghịch thế.

In [23]:
def exercise_5_inversions_by_insertion(values: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_inversions_by_insertion([4, 2, 1, 3]) == 4
# assert exercise_5_inversions_by_insertion([1, 2, 3]) == 0
# assert exercise_5_inversions_by_insertion([3, 2, 1]) == 3

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_insertion` sau khi tự làm.

**Bài 3**
- Mức 1: nút giả trỏ vào `head`; tìm `prev`; chèn; trả `dummy.next`.

**Bài 4**
- Mức 1: nếu rỗng: tạo nút tự trỏ.
- Mức 2: `p = head`; lặp: nếu `p.val <= x <= p.next.val` → chèn; nếu `p.val > p.next.val` và (`x >= p.val` hoặc `x <= p.next.val`) → chèn; nếu đi hết vòng (`p.next is head`) → chèn đại (mọi giá trị bằng nhau).

**Bài 5**
- Mức 1: sau khi tìm được `prev`, số nút lớn hơn `cur` = (kích thước phần đã sắp xếp) − (số bước đã đi tới `prev`).

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
_ = trace_insertion([5, 1, 4, 2, 3])

chèn  5 vào đầu     (0 bước tìm) -> đã sắp xếp: [5] | còn lại: [1, 4, 2, 3]
chèn  1 vào đầu     (0 bước tìm) -> đã sắp xếp: [1, 5] | còn lại: [4, 2, 3]
chèn  4 vào sau 1   (1 bước tìm) -> đã sắp xếp: [1, 4, 5] | còn lại: [2, 3]
chèn  2 vào sau 1   (1 bước tìm) -> đã sắp xếp: [1, 2, 4, 5] | còn lại: [3]
chèn  3 vào sau 2   (2 bước tìm) -> đã sắp xếp: [1, 2, 3, 4, 5] | còn lại: []


## Đáp án Bài 2

In [25]:
def answer_2_insertion_sort(head: Optional[ListNode]) -> Optional[ListNode]:
    return insertion_sort_basic(head)

assert to_list(answer_2_insertion_sort(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_list(answer_2_insertion_sort(build_list([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]
print("Bài 2 PASS (giống insertion_sort_basic ở mục 4)")

Bài 2 PASS (giống insertion_sort_basic ở mục 4)


## Đáp án Bài 3

In [26]:
def answer_3_insert_sorted(head: Optional[ListNode], x: int) -> ListNode:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.val <= x:
        prev = prev.next
    prev.next = ListNode(x, prev.next)
    return dummy.next

assert to_list(answer_3_insert_sorted(build_list([1, 3, 5]), 4)) == [1, 3, 4, 5]
assert to_list(answer_3_insert_sorted(build_list([1, 3, 5]), 0)) == [0, 1, 3, 5]
assert to_list(answer_3_insert_sorted(None, 7)) == [7]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_insert_circular(head: Optional[ListNode], x: int) -> ListNode:
    node = ListNode(x)
    if head is None:
        node.next = node
        return node
    p = head
    while True:
        if p.val <= x <= p.next.val:
            break
        if p.val > p.next.val and (x >= p.val or x <= p.next.val):
            break
        if p.next is head:
            break
        p = p.next
    node.next = p.next
    p.next = node
    return head

h = build_list([3, 4, 1]); h.next.next.next = h; r = answer_4_insert_circular(h, 2); assert sorted(circ_to_list(r, 4)) == [1, 2, 3, 4] and circ_to_list(r, 4) == [3, 4, 1, 2]
r = answer_4_insert_circular(None, 1); assert r.next is r and r.val == 1
for _ in range(2000):
    vals = sorted(random.randint(0, 5) for _ in range(random.randint(1, 6)))
    k = random.randrange(len(vals))
    rot = vals[k:] + vals[:k]
    h = build_list(rot)
    n = h
    while n.next: n = n.next
    n.next = h
    x = random.randint(-1, 6)
    r = answer_4_insert_circular(h, x)
    seq = circ_to_list(r, len(vals) + 1)
    breaks = sum(seq[i] > seq[(i + 1) % len(seq)] for i in range(len(seq)))
    assert sorted(seq) == sorted(vals + [x]) and breaks <= 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_inversions_by_insertion(values: List[int]) -> int:
    dummy, size, inv = ListNode(), 0, 0
    for v in values:
        prev, steps = dummy, 0
        while prev.next and prev.next.val <= v:
            prev = prev.next
            steps += 1
        inv += size - steps
        prev.next = ListNode(v, prev.next)
        size += 1
    return inv

assert answer_5_inversions_by_insertion([4, 2, 1, 3]) == 4
assert answer_5_inversions_by_insertion([1, 2, 3]) == 0
assert answer_5_inversions_by_insertion([3, 2, 1]) == 3
for _ in range(2000):
    a = [random.randint(0, 6) for _ in range(random.randint(0, 10))]
    assert answer_5_inversions_by_insertion(a) == sum(a[i] > a[j] for i in range(len(a)) for j in range(i + 1, len(a)))
print("Bài 5 PASS")

Bài 5 PASS

# 14. Tổng kết

> Nút giả làm đầu danh sách đã sắp xếp; lấy từng nút (nhớ **lưu `next`** trước), đi xuôi tìm `prev` cuối cùng có `val ≤ cur.val`, chèn sau `prev`. Thêm con trỏ `tail` để dữ liệu gần sắp xếp chạy `O(n)`. Ổn định, `O(1)` bộ nhớ, xấu nhất `O(n²)`.

---

## Checklist trước khi nộp

- [ ] Lưu `nxt = cur.next` trước khi chèn?
- [ ] So sánh `<=` để ổn định?
- [ ] Gắn vào đuôi: đặt `tail.next = None`?
- [ ] Trả `dummy.next`?